# SEXTANTE: text mining and occupational segments

This notebook runs the same analysis functions on a synthetic Colombian vacancies sample by default. Set `SEXTANTE_USE_HF=1` to read a small prefix of the private Hugging Face corpus instead. It never modifies the canonical dataset.

Install `requirements-notebook-experiments.txt` to run every cell. HF mode additionally needs `HF_TOKEN` and a Spanish ESCO v1.2.1 CSV from the [official ESCO portal](https://esco.ec.europa.eu/en/use-esco/download), selected with `ESCO_CSV`.

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(repo_root))
from sextante_analysis.eda import summarize

if os.environ.get('SEXTANTE_USE_HF') == '1':
    from sextante_analysis.corpus import load_corpus
    esco_csv = Path(os.environ['ESCO_CSV'])
    postings, source = load_corpus(limit=3000)
else:
    esco_csv = Path('/tmp/sextante-esco-demo.csv')
    esco_csv.write_text('conceptUri,conceptType,preferredLabel,altLabels\n'
        'https://example.org/esco/1,Skill/competence,Python,programación en Python\n'
        'https://example.org/esco/2,Skill/competence,SQL,consultas SQL\n'
        'https://example.org/esco/3,Skill/competence,atención al cliente,servicio al cliente\n', encoding='utf-8')
    role_text = [
        ('Data analyst', 'Python SQL analysis dashboards and data quality for business decisions.'),
        ('Customer support', 'Customer service communication case management and support ticket resolution.'),
        ('Maintenance technician', 'Preventive maintenance electrical safety equipment inspection and repair.'),
    ]
    postings = pd.DataFrame([
        {'vacancy_id': f'demo-{index:03d}', 'source': ('spe' if index % 2 else 'linkedin'),
         'url': f'https://example.org/jobs/{index}', 'title': role_text[index % 3][0],
         'description': 'We are hiring a professional. Required skills: ' + role_text[index % 3][1] * 3,
         'salary_min': 2500000 + (index % 3) * 200000, 'salary_max': 4000000 + (index % 3) * 200000,
         'department': ('Bogotá D.C.' if index % 2 else 'Antioquia'), 'captured_at': '2026-10-08 12:00:00'}
        for index in range(48)
    ])
    source = {'repo': 'synthetic/demo', 'revision': 'fixture-v1', 'rows': len(postings)}
source, postings.shape

({'repo': 'synthetic/demo', 'revision': 'fixture-v1', 'rows': 48}, (48, 9))

In [2]:
pd.Series(summarize(postings)['missing_fraction']).sort_values(ascending=False).to_frame('missing_fraction')

,missing_fraction
vacancy_id,0.0
source,0.0
url,0.0
title,0.0
description,0.0
salary_min,0.0
salary_max,0.0
department,0.0
captured_at,0.0


In [3]:
from sextante_analysis.taxonomy import extract_skills, load_esco_csv
esco = load_esco_csv(esco_csv)
skill_mentions = extract_skills(postings, esco)
skill_mentions['skill_label'].value_counts().head(20)

skill_label
Python    16
SQL       16
Name: count, dtype: int64

In [4]:
from sextante_analysis.duplicates import find_duplicate_candidates
duplicate_candidates = find_duplicate_candidates(postings, threshold=0.90)
duplicate_candidates.head(10)

,vacancy_id_a,source_a,url_a,title_a,vacancy_id_b,source_b,url_b,title_b,similarity
0,demo-000,linkedin,https://example.org/jobs/0,Data analyst,demo-003,spe,https://example.org/jobs/3,Data analyst,1.0
1,demo-000,linkedin,https://example.org/jobs/0,Data analyst,demo-006,linkedin,https://example.org/jobs/6,Data analyst,1.0
2,demo-000,linkedin,https://example.org/jobs/0,Data analyst,demo-009,spe,https://example.org/jobs/9,Data analyst,1.0
3,demo-000,linkedin,https://example.org/jobs/0,Data analyst,demo-012,linkedin,https://example.org/jobs/12,Data analyst,1.0
4,demo-000,linkedin,https://example.org/jobs/0,Data analyst,demo-015,spe,https://example.org/jobs/15,Data analyst,1.0
5,demo-000,linkedin,https://example.org/jobs/0,Data analyst,demo-018,linkedin,https://example.org/jobs/18,Data analyst,1.0
6,demo-000,linkedin,https://example.org/jobs/0,Data analyst,demo-021,spe,https://example.org/jobs/21,Data analyst,1.0
7,demo-000,linkedin,https://example.org/jobs/0,Data analyst,demo-024,linkedin,https://example.org/jobs/24,Data analyst,1.0
8,demo-000,linkedin,https://example.org/jobs/0,Data analyst,demo-027,spe,https://example.org/jobs/27,Data analyst,1.0
9,demo-000,linkedin,https://example.org/jobs/0,Data analyst,demo-030,linkedin,https://example.org/jobs/30,Data analyst,1.0


In [5]:
from pathlib import Path
from sextante_analysis.features import run_models
demo_dir = Path('/tmp/sextante-analysis-demo')
model_summary = run_models(postings, demo_dir, topic_count=8, cluster_count=6)
model_summary

/mnt/c/Users/Shalo/Desktop/SEXTANTE/.venv/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:125: RuntimeWarning: invalid value encountered in sqrt
  return np.sqrt(res * 2)


/mnt/c/Users/Shalo/Desktop/SEXTANTE/.venv/lib/python3.12/site-packages/sklearn/base.py:1403: ConvergenceWarning: Number of distinct clusters (3) found smaller than n_clusters (6). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


{'text_rows': 48,
 'tfidf_features': 62,
 'topic_count': 8,
 'cluster_count': 3,
 'cluster_count_requested': 6,
 'silhouette_cosine_sample': 0.9999998211860657,
 'silhouette_sample_rows': 48,
 'projection': 'TruncatedSVD-2D (sparse PCA approximation)',
 'embedding_model': None}

In [6]:
pd.read_csv(demo_dir / 'topics.csv')

,topic_id,terms
0,0,support | customer | support ticket | service ...
1,1,maintenance | maintenance electrical | prevent...
2,2,data | sql | python sql | python | quality | s...
3,3,maintenance | maintenance electrical | prevent...
4,4,maintenance technician | technician | skills p...
5,5,maintenance electrical | preventive maintenanc...
6,6,technician | technician we | skills preventive...
7,7,technician | technician we | skills preventive...


In [7]:
pd.read_csv(demo_dir / 'segment_profiles.csv')

,cluster_id,postings,terms
0,0,16,maintenance | maintenance electrical | prevent...
1,1,16,data | sql | python sql | python | quality | s...
2,2,16,support | customer | support ticket | service ...


## Optional notebook comparisons

Install `requirements-notebook-experiments.txt` for Word2Vec, UMAP, and t-SNE. These small-sample comparisons are exploratory and are not part of the batch artifact contract.

In [8]:
import numpy as np
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from sextante_analysis.features import prepare_descriptions
sentences = [simple_preprocess(text, deacc=True) for text in prepare_descriptions(postings)]
word2vec = Word2Vec(sentences=sentences, vector_size=100, window=5, min_count=3, workers=1, seed=42)
word_vectors = word2vec.wv
document_vectors = []
for tokens in sentences:
    known = [word_vectors[token] for token in tokens if token in word_vectors]
    document_vectors.append(sum(known) / len(known) if known else [0.0] * 100)
document_vectors = np.asarray(document_vectors, dtype='float32')
document_vectors.shape

(48, 100)

In [9]:
import numpy as np
import umap
from sklearn.cluster import AgglomerativeClustering, DBSCAN
from sklearn.manifold import TSNE
valid = np.flatnonzero(np.linalg.norm(document_vectors, axis=1) > 0)[:1000]
sample_vectors = document_vectors[valid]
umap_projection = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, metric='cosine', random_state=42).fit_transform(sample_vectors)
tsne_projection = TSNE(n_components=2, perplexity=30, init='pca', learning_rate='auto', random_state=42).fit_transform(sample_vectors)
hierarchical_labels = AgglomerativeClustering(n_clusters=6).fit_predict(sample_vectors)
density_labels = DBSCAN(eps=0.5, min_samples=5, metric='cosine').fit_predict(sample_vectors)
{'umap_shape': umap_projection.shape, 'tsne_shape': tsne_projection.shape, 'hierarchical_groups': int(np.unique(hierarchical_labels).size), 'dbscan_labels': sorted(np.unique(density_labels).tolist())}

/mnt/c/Users/Shalo/Desktop/SEXTANTE/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/mnt/c/Users/Shalo/Desktop/SEXTANTE/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


{'umap_shape': (48, 2),
 'tsne_shape': (48, 2),
 'hierarchical_groups': 6,
 'dbscan_labels': [0]}

## Interpreting the results

ESCO matches are exact label/alias mentions and may contain contextual false positives. Duplicate rows are review candidates above the configured threshold. NMF topics and K-Means groups are exploratory summaries, not validated occupational classifications. Batch artifacts record the source revision and model parameters.